https://youtu.be/lD7BCrMtxwI

In [0]:
data = [
(1, '2024-03-01'),
(1, '2024-03-02'),
(1, '2024-03-03'),
(1, '2024-03-04'),
(1, '2024-03-06'),
(1, '2024-03-10'),
(1, '2024-03-11'),
(1, '2024-03-12'),
(1, '2024-03-13'),
(1, '2024-03-14'),
(1, '2024-03-20'),
(1, '2024-03-25'),
(1, '2024-03-26'),
(1, '2024-03-27'),
(1, '2024-03-28'),
(1, '2024-03-29'),
(1, '2024-03-30'),
(2, '2024-03-01'),
(2, '2024-03-02'),
(2, '2024-03-03'),
(2, '2024-03-04'),
(3, '2024-03-01'),
(3, '2024-03-02'),
(3, '2024-03-03'),
(3, '2024-03-04'),
(3, '2024-03-04'),
(3, '2024-03-04'),
(3, '2024-03-05'),
(4, '2024-03-01'),
(4, '2024-03-02'),
(4, '2024-03-03'),
(4, '2024-03-04'),
(4, '2024-03-04')
]

schema = "user_id int , login_date string"

df = spark.createDataFrame(data = data , schema = schema)
df.display()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

In [0]:
df1 = (
    df.withColumn("rank", F.dense_rank().over(Window.partitionBy("user_id").orderBy("user_id","Login_date")))
        #.withColumn("prev_date", F.when(F.col("prev_date").isNull()  ,0).otherwise(F.col("prev_date")))
        .withColumn("diff", F.to_date(F.col("login_date")) - F.col("rank"))
        .groupBy("user_id","diff")
           .agg(F.count("diff").alias("count"),F.collect_list("login_date").alias("list"))
           #.agg(F.collect_list("login_date").alias("list"))
        .filter("count > 4")
        .withColumns( {
            "start_date" :  F.col("list")[0],
            "end_date" : F.col("list")[F.col("count") - 1]
        })
        .drop("list","diff")
        #.withColumn("end_date", F.date_add(F.col("diff") - 1, F.col("count").cast("int")))
        
)
df1.display()